# Mega Project 3 — Risk Segmentation
## Problem 4: Revolving Credit Utilization Segmentation — Real Unsupervised Clustering
## Independent of PD Level (Problem 1), External Bureau Behavior (Problem 2), and
## Instalment-Loan Repayment Conduct (Problem 3)

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Beyond risk level (Problem 1), external bureau behavior (Problem 2), and
instalment-loan repayment conduct (Problem 3), a collections or portfolio-
management team also wants to know how an applicant has actually used
revolving credit -- credit cards -- on their own previous Home Credit loans.
This notebook builds that fourth, independent axis.

### This notebook trains no supervised model and scores no PD
It builds a real, vectorized revolving-credit-utilization feature set from
real `credit_card_balance.csv` (the applicant's own actual month-by-month
credit-card balance, credit limit, drawings, and minimum-payment record on
PREVIOUS Home Credit revolving loans), then applies real unsupervised
K-Means clustering -- grouping applicants by REVOLVING-USAGE-PATTERN
SIMILARITY, never trained against real `TARGET`.

### Why this is a genuine, not redundant, fourth axis
Problem 1 tiers applicants by real PD LEVEL. Problem 2 clusters applicants
by real EXTERNAL bureau behavior. Problem 3 clusters applicants by real
INSTALMENT-LOAN repayment conduct. This notebook touches none of those
tables. Its real feature set is built entirely from the applicant's own
real REVOLVING/credit-card usage on PREVIOUS Home Credit loans -- real
utilization level, real minimum-payment-only behavior, and real
cash-advance frequency -- a genuinely different real signal. Real, computed
Cramer's V against Problem 1's Risk Tier and (when available) Problem 2's
Bureau Segment and Problem 3's Repayment Segment are reported as honest
evidence of how independent this axis actually turned out to be -- not an
asserted claim.

### Hard and soft dependencies
Hard dependency: this notebook requires Mega Project 3 / Notebook 01's real
per-applicant output (`PD`, `TARGET`, `RISK_TIER`) to already exist -- PD is
reused unchanged, never re-scored here. Soft dependencies: Notebook 02's
real Bureau Segment output and Notebook 03's real Repayment Segment output,
if present, each enable an additional cross-axis independence check; this
notebook still produces a complete, standalone result if either or both are
absent.

### Advanced error tackling applied (see LESSONS_LEARNED.md for the
### incidents each of these prevents a repeat of)
- HARD dependency on Notebook 01's real output, checked by actual required
  columns present, not just file existence.
- No `monotonic_within_noise()` call in this notebook, by design --
  revolving-usage clusters are unordered categorical groups with no
  expected direction, the same reasoning already established three times
  in this suite (MP2 Notebook 05, MP3 Notebooks 02 and 03).
- No `matplotlib.use(...)` call anywhere in this file -- lets Jupyter's own
  inline backend handle `plt.show()` cleanly.
- Real, disclosed sampling for computational tractability: silhouette
  score uses scikit-learn's own `sample_size` parameter.
- Data-driven K, never a fixed cluster count.
- Applicants with zero real previous-loan revolving-credit history are
  never silently imputed into a cluster with fabricated average values --
  they get their own explicit "No Revolving Credit History" segment.
- Real, disclosed null handling: a real month with no minimum payment due
  has a null `AMT_INST_MIN_REGULARITY` -- dropped from the minimum-payment
  aggregations (never treated as 0, which would fabricate an "underpaid"
  signal), but still counted in the total-months feature.
- Real, disclosed winsorization applied FROM THE START (not discovered the
  hard way): the 9 unbounded real features are clipped to the real
  1st/99th percentile of the with-history population before
  `StandardScaler`, pre-emptively applying Notebook 03's real-data lesson
  so a small number of extreme real values cannot dominate the distance
  K-Means clusters on.
- K range starts at 2 and the stability floor defaults to 1% (not 3%),
  also a pre-emptive application of Notebook 03's real-data diagnosis --
  both remain config-overridable in `project_config.json`.
- Two soft cross-checks (Notebook 02's Bureau Segment, Notebook 03's
  Repayment Segment) in addition to the Problem 1 Risk Tier cross-check --
  the most independence evidence gathered for any MP3 problem so far.
- No EDA section, per standing instruction.

### Verification status
Verified end-to-end on this suite's synthetic fixture via real Jupyter
execution -- 0 errors, all integrity and statistical-robustness checks pass,
HTML dashboard confirmed under a network-blocked Playwright check, Excel
workbook confirmed via LibreOffice headless recalculation. **Not yet run
against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 04 — MEGA PROJECT 3: RISK SEGMENTATION
# PROBLEM 4: REVOLVING CREDIT UTILIZATION SEGMENTATION
# Real Unsupervised Clustering on the Applicant's Own Real Credit-Card Usage
# Pattern on PREVIOUS Home Credit Loans — Independent of PD Level (Problem 1),
# External Bureau Behavior (Problem 2), and Instalment-Loan Repayment Conduct
# (Problem 3)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains no supervised model and
# scores no PD. It builds a real, vectorized revolving-credit-utilization
# feature set from real credit_card_balance.csv (the applicant's own actual
# month-by-month credit-card balance, credit limit, drawings, and minimum-
# payment record on PREVIOUS Home Credit revolving loans) via
# src/features/risk_segmentation_features.py, then applies real unsupervised
# K-Means clustering -- grouping applicants by REVOLVING-USAGE-PATTERN
# SIMILARITY, never trained against real TARGET.
#
# WHY THIS IS A GENUINE, NOT REDUNDANT, FOURTH AXIS (read this before trusting
# any "independent axis" claim below): Problem 1 tiers applicants by real PD
# LEVEL. Problem 2 clusters applicants by real EXTERNAL bureau behavior --
# credit history at OTHER institutions. Problem 3 clusters applicants by real
# INSTALMENT-LOAN repayment conduct (installments_payments.csv /
# POS_CASH_balance.csv). This notebook touches none of those tables. It builds
# its real feature set entirely from the applicant's own real REVOLVING/
# credit-card usage on PREVIOUS Home Credit loans -- utilization level, real
# minimum-payment-only behavior, and real cash-advance frequency -- see
# src/features/risk_segmentation_features.py's own disclosure for the full
# feature list and its overlap disclosure against Mega Project 1's existing
# credit-card SUM features. Section 9 below computes real Cramer's V against
# Problem 1's Risk Tier and, when available, Problem 2's Bureau Segment AND
# Problem 3's Repayment Segment -- honest, computed evidence of how
# independent this axis actually turned out to be, not an asserted claim.
#
# LESSONS APPLIED FROM THIS SUITE'S OWN HARDENING HISTORY (LESSONS_LEARNED.md
# -- every item below cites which real incident it prevents a repeat of):
#   1. HARD DEPENDENCY on Mega Project 3 / Notebook 01's real per-applicant
#      output (PD, TARGET, RISK_TIER), checked by actual required columns
#      present, not just file existence (LESSONS_LEARNED.md #4) -- PD is
#      never re-scored here; Notebook 01's real values are reused unchanged.
#   2. NO `monotonic_within_noise()` CALL IN THIS NOTEBOOK, BY DESIGN:
#      revolving-usage clusters are unordered categorical groups with no
#      expected direction -- the same "does not apply by construction"
#      reasoning already established three times in this suite (MP2
#      Notebook 05, MP3 Notebooks 02 and 03).
#   3. No `matplotlib.use(...)` call anywhere in this file (LESSONS_LEARNED.md
#      #7) -- lets Jupyter's own inline backend handle `plt.show()` cleanly.
#   4. REAL, DISCLOSED SAMPLING FOR COMPUTATIONAL TRACTABILITY: silhouette
#      score uses scikit-learn's own `sample_size` parameter, the same
#      disclosed pattern Notebooks 02 and 03 already established.
#   5. DATA-DRIVEN K, NEVER A FIXED CLUSTER COUNT: the real number of
#      clusters is chosen by the real silhouette score across a documented
#      candidate range -- "achieved, not forced," as in Problems 1-3.
#   6. APPLICANTS WITH ZERO REAL REVOLVING-CREDIT HISTORY ARE NEVER SILENTLY
#      IMPUTED into a cluster with fabricated average values -- they get
#      their own explicit "No Revolving Credit History" segment, disclosed
#      by real, measured prevalence, not hidden.
#   7. REAL NULL HANDLING, DISCLOSED: a real month with no minimum payment
#      due has null AMT_INST_MIN_REGULARITY -- dropped from the minimum-
#      payment aggregations (never treated as 0, which would fabricate an
#      "underpaid" signal), but still counted in N_CC_MONTHS. See
#      src/features/risk_segmentation_features.py's own disclosure.
#   8. WINSORIZATION APPLIED FROM THE START, NOT DISCOVERED THE HARD WAY:
#      Notebook 03's real-data incident (unclipped unbounded features let a
#      handful of extreme real values dominate Euclidean distance and cause
#      K-Means to isolate them as their own tiny outlier cluster) is applied
#      pre-emptively here -- see the winsorization block in
#      engineer_revolving_credit_utilization_features().
#   9. K RANGE STARTS AT 2 AND THE STABILITY FLOOR DEFAULTS TO 1%, NOT 3%:
#      also a direct, pre-emptive application of what Notebook 03's real-data
#      diagnosis established -- a real 307K-scale population may only
#      support a small number of broad, stable segments, and the true
#      minimum-viable-cluster size for real behavioral minority groups can
#      be closer to 1% than 3% of the population. Both defaults remain
#      config-overridable in project_config.json if this notebook's own real
#      run needs different values.
#  10. TWO SOFT CROSS-CHECKS (Problem 2's Bureau Segment, Problem 3's
#      Repayment Segment) in addition to the Problem 1 Risk Tier cross-check
#      -- the most independence evidence gathered for any MP3 problem so
#      far; neither is required for this notebook to produce a complete,
#      standalone result on its own.
#  11. NO EDA SECTION -- per standing instruction.
# ============================================================================

import os
import sys
import json
import time
import warnings
import joblib
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))

MP3_DIR = SUITE_ROOT / "03_mega_project_3_risk_segmentation"
ARTIFACTS_DIR = MP3_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP3_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP3_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, load_csv_cached, check_ram_headroom

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import)
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
T0 = time.time()

from features.risk_segmentation_features import engineer_revolving_credit_utilization_features
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — HARD DEPENDENCY: Mega Project 3 / Notebook 01's real per-
# applicant output, checked by actual required columns (LESSONS_LEARNED.md
# #4). Only 1 raw table is loaded here (credit_card_balance) -- PD is reused
# unchanged from Notebook 01, never re-scored.
# ---------------------------------------------------------------------------
NB01_PATH = ARTIFACTS_DIR / "notebook_01_risk_tiers.csv"
if not NB01_PATH.exists():
    raise FileNotFoundError(
        "Mega Project 3 / Notebook 04 requires Mega Project 3 / Notebook 01's real "
        "per-applicant output, which has not been produced on this machine yet. Fix: run "
        "03_mega_project_3_risk_segmentation/notebooks/01_data_driven_risk_tier_construction.ipynb "
        "end-to-end first, then re-run this notebook."
    )
nb01 = pl.read_csv(NB01_PATH)
_req = ["SK_ID_CURR", "PD", "TARGET", "RISK_TIER"]
_missing = [c for c in _req if c not in nb01.columns]
if _missing:
    raise KeyError(f"Required columns missing from Notebook 01's output: {_missing}. Re-run Notebook 01.")
N_SCOPE = nb01.height
print(f"[LOAD] Real per-applicant output from Notebook 01: {N_SCOPE:,} rows.")

# SOFT dependencies (LESSON #10): Notebook 02's real Bureau Segment and
# Notebook 03's real Repayment Segment outputs, for optional cross-axis
# checks in Section 9. Neither is required.
NB02_SEGMENTS_PATH = ARTIFACTS_DIR / "notebook_02_bureau_segments.csv"
NB02_AVAILABLE = NB02_SEGMENTS_PATH.exists()
if NB02_AVAILABLE:
    nb02_segments = pl.read_csv(NB02_SEGMENTS_PATH).select(["SK_ID_CURR", "BUREAU_SEGMENT"])
    print(f"[SOFT-DEPENDENCY] Real Notebook 02 Bureau Segment output found -- "
          f"{nb02_segments.height:,} rows will be used for an optional cross-axis check.")
else:
    print("[SOFT-DEPENDENCY] Real Notebook 02 Bureau Segment output not found -- this notebook still "
          "produces a complete, standalone result; only the optional Bureau-Segment cross-check is skipped.")

NB03_SEGMENTS_PATH = ARTIFACTS_DIR / "notebook_03_repayment_segments.csv"
NB03_AVAILABLE = NB03_SEGMENTS_PATH.exists()
if NB03_AVAILABLE:
    nb03_segments = pl.read_csv(NB03_SEGMENTS_PATH).select(["SK_ID_CURR", "REPAYMENT_SEGMENT"])
    print(f"[SOFT-DEPENDENCY] Real Notebook 03 Repayment Segment output found -- "
          f"{nb03_segments.height:,} rows will be used for an optional cross-axis check.")
else:
    print("[SOFT-DEPENDENCY] Real Notebook 03 Repayment Segment output not found -- this notebook still "
          "produces a complete, standalone result; only the optional Repayment-Segment cross-check is skipped.")

credit_card = load_csv_cached(RAW_DIR / "credit_card_balance.csv", PARQUET_CACHE_DIR)
check_ram_headroom(PERF)
print(f"[DATA] Real credit_card_balance.csv: {credit_card.shape[0]:,} rows.")

# ---------------------------------------------------------------------------
# SECTION 5 — Real revolving-credit-utilization feature engineering (HYPER
# reuse)
# ---------------------------------------------------------------------------
feat_df, FEATURE_NAMES, WINSORIZE_REPORT = engineer_revolving_credit_utilization_features(
    nb01.select("SK_ID_CURR"), credit_card
)
df = nb01.join(feat_df, on="SK_ID_CURR", how="left").to_pandas()
if NB02_AVAILABLE:
    df = df.merge(nb02_segments.to_pandas(), on="SK_ID_CURR", how="left")
if NB03_AVAILABLE:
    df = df.merge(nb03_segments.to_pandas(), on="SK_ID_CURR", how="left")
N_WITH_HISTORY = int(df["HAS_REVOLVING_HISTORY"].sum())
PCT_WITH_HISTORY = N_WITH_HISTORY / N_SCOPE
print(f"[FEATURES] {len(FEATURE_NAMES)} real revolving-credit-utilization features engineered. "
      f"{N_WITH_HISTORY:,} of {N_SCOPE:,} real applicants ({PCT_WITH_HISTORY:.1%}) have real "
      f"previous-loan revolving-credit history.")
print(f"[FEATURES] Winsorized {len(WINSORIZE_REPORT)} unbounded real features at the 1st/99th "
      f"percentile (computed over applicants WITH real revolving-credit history only) so a small "
      f"number of extreme real values cannot dominate Euclidean distance after StandardScaler:")
for _col, _rep in WINSORIZE_REPORT.items():
    print(f"    {_col}: real range clipped to [{_rep['lo']:.2f}, {_rep['hi']:.2f}] -- "
          f"{_rep['n_clipped_low']:,} real values clipped low, {_rep['n_clipped_high']:,} clipped "
          f"high, of {_rep['n_with_history']:,} real applicants with revolving-credit history.")

# ---------------------------------------------------------------------------
# SECTION 6 — Real, data-driven K-Means clustering (applicants WITH real
# revolving-credit history only -- LESSON #6: never impute the rest into a
# cluster). K range and stability floor pre-emptively widened per LESSON #9.
# ---------------------------------------------------------------------------
with_hist = df[df["HAS_REVOLVING_HISTORY"]].reset_index(drop=True)
X = with_hist[FEATURE_NAMES].to_numpy(dtype=float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

K_RANGE = list(range(int(CONFIG.get("revolving_segment_k_min", 2)), int(CONFIG.get("revolving_segment_k_max", 8)) + 1))
MIN_CLUSTER_FRACTION = float(CONFIG.get("revolving_segment_min_cluster_fraction", 0.01))
MIN_CLUSTER_SIZE = max(int(MIN_CLUSTER_FRACTION * len(with_hist)), 20)
SIL_SAMPLE_SIZE = min(int(CONFIG.get("revolving_segment_silhouette_sample_size", 10_000)), len(with_hist))

k_results = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    counts = np.bincount(labels)
    if counts.min() < MIN_CLUSTER_SIZE:
        print(f"[K-SELECTION] k={k}: rejected -- smallest real cluster ({counts.min():,}) is below the "
              f"minimum stable size ({MIN_CLUSTER_SIZE:,}, {MIN_CLUSTER_FRACTION:.1%} of the real population).")
        continue
    sil = silhouette_score(X_scaled, labels, sample_size=SIL_SAMPLE_SIZE, random_state=SEED)
    k_results.append({"k": k, "silhouette": float(sil), "model": km, "labels": labels})
    print(f"[K-SELECTION] k={k}: real silhouette score={sil:.4f} (sampled {SIL_SAMPLE_SIZE:,} of "
          f"{len(with_hist):,} real applicants for tractability).")

if not k_results:
    raise RuntimeError(
        f"No candidate K in {K_RANGE} produced every real cluster above the minimum stable size "
        f"({MIN_CLUSTER_SIZE:,}) -- the real data does not support this many distinguishable revolving-"
        f"credit-utilization segments at this population size. Lower revolving_segment_k_max or "
        f"revolving_segment_min_cluster_fraction in project_config.json."
    )
best = max(k_results, key=lambda r: r["silhouette"])
K_CHOSEN = best["k"]
SILHOUETTE_CHOSEN = best["silhouette"]
CLUSTER_LABELS_RAW = best["labels"]
print(f"[K-SELECTION] Real data-driven choice: k={K_CHOSEN} (highest real silhouette score "
      f"{SILHOUETTE_CHOSEN:.4f} among {len(k_results)} candidate(s) tried).")

SEGMENT_LABELS = [f"Utilization Segment {chr(65 + i)}" for i in range(K_CHOSEN)]
with_hist = with_hist.copy()
with_hist["UTILIZATION_SEGMENT"] = [SEGMENT_LABELS[i] for i in CLUSTER_LABELS_RAW]

no_hist = df[~df["HAS_REVOLVING_HISTORY"]].copy()
no_hist["UTILIZATION_SEGMENT"] = "No Revolving Credit History"
seg_df = pd.concat([with_hist, no_hist], ignore_index=True)
ALL_SEGMENT_LABELS = SEGMENT_LABELS + ["No Revolving Credit History"]

# ---------------------------------------------------------------------------
# SECTION 7 — Real segment profiling (descriptive, not EDA — this is the
# deliverable output, computed once, after modeling).
# ---------------------------------------------------------------------------
profile_cols = ["MEAN_UTILIZATION", "MAX_UTILIZATION", "PCT_MONTHS_HIGH_UTILIZATION",
                 "MEAN_MIN_PAYMENT_RATIO", "PCT_MONTHS_MIN_PAYMENT_ONLY",
                 "PCT_MONTHS_ANY_CASH_ADVANCE", "MEAN_SK_DPD", "PCT_MONTHS_ACTIVE"]
seg_agg = (
    seg_df.groupby("UTILIZATION_SEGMENT", observed=True)
    .agg(n_applicants=("SK_ID_CURR", "size"), real_default_rate=("TARGET", "mean"),
         **{f"mean_{c.lower()}": (c, "mean") for c in profile_cols})
    .reindex(ALL_SEGMENT_LABELS).reset_index()
)
seg_agg["n_applicants"] = seg_agg["n_applicants"].astype(int)
for _, row in seg_agg.iterrows():
    print(f"[SEGMENT] {row['UTILIZATION_SEGMENT']}: {int(row['n_applicants']):,} real applicants, "
          f"real default rate={row['real_default_rate']:.4f}, "
          f"mean utilization={row['mean_mean_utilization']:.3f}, "
          f"% months any cash advance={row['mean_pct_months_any_cash_advance']:.3f}.")

# ---------------------------------------------------------------------------
# SECTION 8 — Real chi-square + Cramer's V + vectorized bootstrap CI
# (Utilization Segment vs. real TARGET).
# ---------------------------------------------------------------------------
contingency = pd.crosstab(seg_df["UTILIZATION_SEGMENT"], seg_df["TARGET"])
n_obs = int(contingency.values.sum())
min_dim = min(contingency.shape) - 1
chi2_stat, chi2_p, chi2_dof, _ = chi2_contingency(contingency)
cramers_v = float(np.sqrt((chi2_stat / n_obs) / max(min_dim, 1))) if min_dim > 0 else 0.0
print(f"[CHI-SQUARE] Real Utilization Segment vs. real TARGET: chi2={chi2_stat:.2f}, dof={chi2_dof}, "
      f"p-value={chi2_p:.6g}, Cramer's V={cramers_v:.4f}.")

N_BOOTSTRAP = 500
cell_probs = (contingency.values / n_obs).flatten()
cell_shape = contingency.shape
boot_v = []
for _ in range(N_BOOTSTRAP):
    draw = rng.multinomial(n_obs, cell_probs).reshape(cell_shape)
    if draw.sum() == 0 or min(draw.shape) < 2:
        continue
    try:
        chi2_bs, _, _, _ = chi2_contingency(draw)
        md_bs = min(draw.shape) - 1
        boot_v.append(float(np.sqrt((chi2_bs / n_obs) / max(md_bs, 1))) if md_bs > 0 else 0.0)
    except ValueError:
        continue
boot_v = np.array(boot_v) if boot_v else np.array([cramers_v])
V_CI_LOW, V_CI_HIGH = float(np.percentile(boot_v, 2.5)), float(np.percentile(boot_v, 97.5))
CRAMERS_V_ROBUST_THRESHOLD = 0.05
print(f"[VALIDATION] Real {len(boot_v)}-resample vectorized bootstrap 95% CI on Cramer's V "
      f"(Utilization Segment vs. real default): [{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}].")

# ---------------------------------------------------------------------------
# SECTION 9 — Real cross-checks: how independent is this segmentation from
# Problem 1's PD-tier segmentation, and (soft dependencies permitting)
# Problem 2's Bureau Segment and Problem 3's Repayment Segment? (Descriptive,
# not gated -- genuine honest evidence.)
# ---------------------------------------------------------------------------
cross_contingency = pd.crosstab(seg_df["UTILIZATION_SEGMENT"], seg_df["RISK_TIER"])
cn_obs = int(cross_contingency.values.sum())
cmin_dim = min(cross_contingency.shape) - 1
cchi2_stat, cchi2_p, _, _ = chi2_contingency(cross_contingency)
CROSS_CRAMERS_V_TIER = float(np.sqrt((cchi2_stat / cn_obs) / max(cmin_dim, 1))) if cmin_dim > 0 else 0.0
print(f"[CROSS-CHECK] Real association between Utilization Segment and Problem 1's Risk Tier: "
      f"Cramer's V={CROSS_CRAMERS_V_TIER:.4f} (lower = more independent axes; reported honestly, "
      f"not gated pass/fail).")

CROSS_CRAMERS_V_BUREAU = None
if NB02_AVAILABLE:
    both_present = seg_df["BUREAU_SEGMENT"].notna()
    bureau_contingency = pd.crosstab(seg_df.loc[both_present, "UTILIZATION_SEGMENT"],
                                      seg_df.loc[both_present, "BUREAU_SEGMENT"])
    bn_obs = int(bureau_contingency.values.sum())
    bmin_dim = min(bureau_contingency.shape) - 1
    bchi2_stat, bchi2_p, _, _ = chi2_contingency(bureau_contingency)
    CROSS_CRAMERS_V_BUREAU = float(np.sqrt((bchi2_stat / bn_obs) / max(bmin_dim, 1))) if bmin_dim > 0 else 0.0
    print(f"[CROSS-CHECK] Real association between Utilization Segment and Problem 2's Bureau Segment: "
          f"Cramer's V={CROSS_CRAMERS_V_BUREAU:.4f} -- these are built from entirely different real tables "
          f"(previous Home Credit revolving loans vs. external bureau history), so a low value would "
          f"evidence these are genuinely different real axes, not a relabeling.")
else:
    print("[CROSS-CHECK] Problem 2's Bureau Segment output not available -- skipping that cross-check "
          "(soft dependency; this notebook's own result is unaffected).")

CROSS_CRAMERS_V_REPAYMENT = None
if NB03_AVAILABLE:
    both_present_r = seg_df["REPAYMENT_SEGMENT"].notna()
    repay_contingency = pd.crosstab(seg_df.loc[both_present_r, "UTILIZATION_SEGMENT"],
                                     seg_df.loc[both_present_r, "REPAYMENT_SEGMENT"])
    rn_obs = int(repay_contingency.values.sum())
    rmin_dim = min(repay_contingency.shape) - 1
    rchi2_stat, rchi2_p, _, _ = chi2_contingency(repay_contingency)
    CROSS_CRAMERS_V_REPAYMENT = float(np.sqrt((rchi2_stat / rn_obs) / max(rmin_dim, 1))) if rmin_dim > 0 else 0.0
    print(f"[CROSS-CHECK] Real association between Utilization Segment and Problem 3's Repayment Segment: "
          f"Cramer's V={CROSS_CRAMERS_V_REPAYMENT:.4f} -- these are built from entirely different real tables "
          f"(previous Home Credit revolving loans vs. previous Home Credit instalment loans), so a low "
          f"value would evidence these are genuinely different real axes, not a relabeling.")
else:
    print("[CROSS-CHECK] Problem 3's Repayment Segment output not available -- skipping that cross-check "
          "(soft dependency; this notebook's own result is unaffected).")

# ---------------------------------------------------------------------------
# SECTION 10 — STATISTICAL ROBUSTNESS VERDICT. NOTE: no monotonicity check
# here by design (LESSON #2 above — unordered categorical segments).
# ---------------------------------------------------------------------------
validation_checks = [
    ("chi_square_significant", chi2_p < 0.05),
    ("cramers_v_ci_excludes_zero", V_CI_LOW > CRAMERS_V_ROBUST_THRESHOLD),
    ("silhouette_score_finite_and_positive", bool(np.isfinite(SILHOUETTE_CHOSEN) and SILHOUETTE_CHOSEN > 0.0)),
    ("every_segment_at_least_min_size", bool((seg_agg["n_applicants"] >= min(MIN_CLUSTER_SIZE, N_SCOPE - N_WITH_HISTORY + 1)).all()
                                              if N_WITH_HISTORY < N_SCOPE else (seg_agg["n_applicants"] >= MIN_CLUSTER_SIZE).all())),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "STATISTICALLY ROBUST — RECOMMENDED FOR PRODUCTION" if ANALYSIS_ROBUST
    else "NOT YET STATISTICALLY ROBUST — failed: " + ", ".join(_failed_validation_checks) +
         " (a separate, stricter statistical-significance gate, distinct from the structural "
         "pipeline integrity checks reported elsewhere in this notebook's output)"
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Statistical robustness verdict: {ANALYSIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 11 — Inline charts. No matplotlib.use(...) call (LESSON #3).
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].bar(seg_agg["UTILIZATION_SEGMENT"].astype(str), seg_agg["real_default_rate"], color=_palette(len(seg_agg)))
axes[0].set_ylabel("Real Default Rate"); axes[0].set_title("Real Default Rate by Utilization Segment")
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")
axes[1].bar(seg_agg["UTILIZATION_SEGMENT"].astype(str), seg_agg["n_applicants"], color=_palette(len(seg_agg)))
axes[1].set_ylabel("Real Applicants"); axes[1].set_title("Real Population by Utilization Segment")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
k_vals = [r["k"] for r in k_results]
sil_vals = [r["silhouette"] for r in k_results]
axes[2].plot(k_vals, sil_vals, marker="o", color=VIVID_PALETTE[0])
axes[2].axvline(K_CHOSEN, color=VIVID_PALETTE[1], linestyle="--", label=f"Chosen k={K_CHOSEN}")
axes[2].set_xlabel("k (candidate cluster count)"); axes[2].set_ylabel("Real Silhouette Score")
axes[2].set_title("Real Data-Driven K Selection"); axes[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_04_utilization_segments.png", dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 12 — Pipeline Integrity Checks (structural)
# ---------------------------------------------------------------------------
checks = [
    ("real_data_loaded", N_SCOPE > 0),
    ("required_columns_present", len(_missing) == 0),
    ("every_applicant_assigned_a_segment", bool(seg_df["UTILIZATION_SEGMENT"].notna().all())),
    ("segment_count_matches_k_plus_no_history", seg_df["UTILIZATION_SEGMENT"].nunique() == K_CHOSEN + (1 if N_WITH_HISTORY < N_SCOPE else 0)),
    ("no_history_applicants_never_clustered", bool((seg_df.loc[~seg_df["HAS_REVOLVING_HISTORY"], "UTILIZATION_SEGMENT"] == "No Revolving Credit History").all())),
    ("contingency_row_count_matches", contingency.shape[0] == len(ALL_SEGMENT_LABELS) - (1 if N_WITH_HISTORY == N_SCOPE else 0)),
    ("chi2_pvalue_in_bounds", 0.0 <= chi2_p <= 1.0),
    ("bootstrap_ci_computed", len(boot_v) > 0),
    ("cpu_thread_ceiling_applied_before_import", os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Pipeline integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 13 — Reporting & Packaging (SOP Stage 5)
# ---------------------------------------------------------------------------
csv_paths = write_csv_outputs(
    {"notebook_04_utilization_segment_aggregation": seg_agg,
     "notebook_04_k_selection": pd.DataFrame(k_results)[["k", "silhouette"]]},
    REPORTS_DIR,
)
seg_df[["SK_ID_CURR", "HAS_REVOLVING_HISTORY", "UTILIZATION_SEGMENT"]].to_csv(
    ARTIFACTS_DIR / "notebook_04_utilization_segments.csv", index=False
)

ASSUMPTIONS = {"K_CHOSEN": K_CHOSEN, "MIN_CLUSTER_FRACTION": MIN_CLUSTER_FRACTION,
               "SILHOUETTE_SAMPLE_SIZE": SIL_SAMPLE_SIZE, "WINSORIZE_PERCENTILE": 0.01}
ASSUMPTION_NOTES = {
    "K_CHOSEN": "Real number of revolving-credit-utilization clusters, chosen by the highest real "
                f"silhouette score among candidates k={K_RANGE} -- never fixed by hand.",
    "MIN_CLUSTER_FRACTION": "Minimum real population fraction per cluster (defaults to 1%, not 3%, "
                             "per Notebook 03's real-data lesson) -- candidate k values producing a "
                             "smaller real cluster are rejected before silhouette scoring.",
    "SILHOUETTE_SAMPLE_SIZE": "Real applicants sampled for the O(n^2) silhouette computation -- a real, "
                               "standard scikit-learn mitigation for computational tractability at scale, "
                               "not a change to the real clustering itself (KMeans still fits on all real "
                               "applicants with revolving-credit history).",
    "WINSORIZE_PERCENTILE": f"{len(WINSORIZE_REPORT)} unbounded real features clipped to the 1st/99th "
                             "percentile (computed over applicants with real revolving-credit history "
                             "only) before StandardScaler, applied from the start per Notebook 03's "
                             "real-data lesson -- bounds, never invents, real values. See "
                             "src/features/risk_segmentation_features.py.",
}

STORY_DEFAULT_RATE = [
    f"Real default rate spans {seg_agg['real_default_rate'].min():.1%} to "
    f"{seg_agg['real_default_rate'].max():.1%} across {len(ALL_SEGMENT_LABELS)} real revolving-credit "
    f"utilization segments (chi-square p={chi2_p:.4g}, Cramer's V={cramers_v:.4f}, 95% bootstrap CI "
    f"[{V_CI_LOW:.4f}, {V_CI_HIGH:.4f}]).",
    f"Cross-check against Problem 1's Risk Tier: Cramer's V={CROSS_CRAMERS_V_TIER:.4f}" +
    (f"; against Problem 2's Bureau Segment: Cramer's V={CROSS_CRAMERS_V_BUREAU:.4f}" if NB02_AVAILABLE else "") +
    (f"; against Problem 3's Repayment Segment: Cramer's V={CROSS_CRAMERS_V_REPAYMENT:.4f}." if NB03_AVAILABLE else "."),
]
INSIGHTS = [{
    "headline": f"{K_CHOSEN} real, data-driven revolving-credit utilization segments found "
                f"(silhouette={SILHOUETTE_CHOSEN:.3f})",
    "specific": STORY_DEFAULT_RATE[0],
    "measurable": f"{N_WITH_HISTORY:,} of {N_SCOPE:,} real applicants ({PCT_WITH_HISTORY:.1%}) clustered; "
                  f"{N_SCOPE - N_WITH_HISTORY:,} real applicants with no previous-loan revolving-credit "
                  f"history reported as their own explicit segment.",
    "achievable": f"Computed end-to-end in {round(time.time() - T0, 1)}s.",
    "relevant": "Gives a collections or portfolio-management team a real revolving-credit-usage axis, "
                "built from the applicant's own conduct on previous Home Credit credit-card loans, "
                "independent of PD level, external bureau behavior, and instalment-loan repayment "
                "conduct, to differentiate treatment by.",
    "timebound": "Re-run after any Notebook 01 re-run (refreshes PD/TARGET/RISK_TIER) or when new real "
                 "credit-card data becomes available.",
}]

word_sections = [{
    "heading": "Real Default Rate by Utilization Segment",
    "paragraphs": ["Real default rate and real population per segment, including applicants with no real "
                   "previous-loan revolving-credit history as their own explicit segment."],
    "table": {"headers": ["Segment", "N Applicants", "Real Default Rate", "Mean Utilization",
                           "% Months Any Cash Advance"],
               "rows": [[r["UTILIZATION_SEGMENT"], f"{int(r['n_applicants']):,}", f"{r['real_default_rate']:.2%}",
                         f"{r['mean_mean_utilization']:.1%}", f"{r['mean_pct_months_any_cash_advance']:.1%}"]
                        for _, r in seg_agg.iterrows()]},
    "image_path": ARTIFACTS_DIR / "notebook_04_utilization_segments.png", "story": STORY_DEFAULT_RATE,
}]
word_path = build_word_report(
    REPORTS_DIR / "notebook_04_report.docx",
    title="Mega Project 3 — Notebook 04: Revolving Credit Utilization Segmentation",
    subtitle=f"Real K-Means clustering, k={K_CHOSEN} (data-driven) — independent of PD level, bureau "
             f"behavior, and repayment conduct",
    exec_summary=[
        f"{N_SCOPE:,} real applicants; {N_WITH_HISTORY:,} ({PCT_WITH_HISTORY:.1%}) have real previous-loan "
        f"revolving-credit history.",
        f"{K_CHOSEN} real data-driven utilization segments found (silhouette={SILHOUETTE_CHOSEN:.3f}).",
        f"Statistical robustness verdict: {ANALYSIS_VERDICT}",
        f"Cross-axis independence from Problem 1's Risk Tier: Cramer's V={CROSS_CRAMERS_V_TIER:.4f}." +
        (f" From Problem 2's Bureau Segment: Cramer's V={CROSS_CRAMERS_V_BUREAU:.4f}." if NB02_AVAILABLE else "") +
        (f" From Problem 3's Repayment Segment: Cramer's V={CROSS_CRAMERS_V_REPAYMENT:.4f}." if NB03_AVAILABLE else ""),
    ],
    sections=word_sections, insights=INSIGHTS,
)

excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_04_workbook.xlsx",
    assumptions=ASSUMPTIONS, assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "Segment Aggregation", "headers": list(seg_agg.columns),
         "rows": seg_agg.astype(object).values.tolist(), "highlight_col": "real_default_rate"},
        {"name": "K Selection", "headers": ["k", "silhouette"],
         "rows": [[r["k"], r["silhouette"]] for r in k_results]},
    ],
    insights_sheet={"name": "SMART Insights", "items": INSIGHTS},
)

kpi_cards = [
    {"label": "Real Applicants", "value": f"{N_SCOPE:,}"},
    {"label": "With Real Revolving-Credit History", "value": f"{PCT_WITH_HISTORY:.1%}"},
    {"label": "Real Data-Driven Segments", "value": str(K_CHOSEN)},
    {"label": "Statistical Verdict", "value": "ROBUST" if ANALYSIS_ROBUST else "NOT YET ROBUST"},
]
charts = [
    {"id": "defaultRateBySegment", "title": "Real Default Rate by Utilization Segment", "type": "bar",
     "labels": seg_agg["UTILIZATION_SEGMENT"].astype(str).tolist(),
     "datasets": [{"label": "Real Default Rate", "data": seg_agg["real_default_rate"].tolist(),
                   "backgroundColor": _palette(len(seg_agg))}], "story": STORY_DEFAULT_RATE},
    {"id": "populationBySegment", "title": "Real Population by Utilization Segment", "type": "bar",
     "labels": seg_agg["UTILIZATION_SEGMENT"].astype(str).tolist(),
     "datasets": [{"label": "Real Applicants", "data": seg_agg["n_applicants"].tolist(),
                   "backgroundColor": _palette(len(seg_agg))}]},
]
html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_04_dashboard.html",
    title="Mega Project 3 — Revolving Credit Utilization Segmentation",
    subtitle=f"{N_SCOPE:,} real applicants — {K_CHOSEN} real data-driven utilization segments",
    kpi_cards=kpi_cards, charts=charts, insights=INSIGHTS,
    data_table={"title": "Segment Aggregation (real)", "columns": list(seg_agg.columns),
                "rows": seg_agg.values.tolist(), "filter_column": "UTILIZATION_SEGMENT"},
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s).")

# ---------------------------------------------------------------------------
# SECTION 13b — Persist the real, fitted clustering bundle (hardening pass):
# the real chosen KMeans model + real fitted StandardScaler + real winsorize
# bounds + real feature list + real segment labels, so a real deployable
# service can assign a NEW real applicant to a segment identically to how
# this notebook assigns its own scored population (winsorize with the same
# saved bounds, scale with the same fitted scaler, predict with the same
# fitted model) -- without this, "deployable segmentation" would not be
# possible without either retraining or fabricating an assignment.
# ---------------------------------------------------------------------------
SEGMENT_MODEL_PATH = ARTIFACTS_DIR / "notebook_04_segment_model.joblib"
joblib.dump({
    "kmeans": best["model"], "scaler": scaler, "feature_names": FEATURE_NAMES,
    "segment_labels": SEGMENT_LABELS, "k_chosen": K_CHOSEN, "random_seed": SEED,
    "winsorize_report": WINSORIZE_REPORT,
}, SEGMENT_MODEL_PATH)
print(f"[ARTIFACT] Real fitted clustering bundle saved: {SEGMENT_MODEL_PATH.name} "
      f"(kmeans, scaler, {len(FEATURE_NAMES)} feature names, {K_CHOSEN} segment labels, "
      f"{len(WINSORIZE_REPORT)} winsorize bounds).")

# ---------------------------------------------------------------------------
# SECTION 14 — Save artifacts + governance stamp (idempotent)
# ---------------------------------------------------------------------------
summary = {
    "notebook": "04_revolving_credit_utilization_segmentation",
    "mega_project": "Mega Project 3 - Risk Segmentation",
    "problem": "Problem 4 - Revolving Credit Utilization Segmentation",
    "random_seed": SEED,
    "n_applicants": N_SCOPE,
    "n_with_revolving_history": N_WITH_HISTORY,
    "pct_with_revolving_history": PCT_WITH_HISTORY,
    "upstream_dependency": {"source_notebook": "Mega Project 3 / Notebook 01", "reused_not_recomputed": True,
                             "columns_reused": _req},
    "soft_dependencies": {"bureau_segment": {"source_notebook": "Mega Project 3 / Notebook 02", "available": NB02_AVAILABLE},
                           "repayment_segment": {"source_notebook": "Mega Project 3 / Notebook 03", "available": NB03_AVAILABLE}},
    "winsorization": {"percentile": 0.01, "applied_to": list(WINSORIZE_REPORT.keys()),
                      "report": WINSORIZE_REPORT},
    "clustering_config": {"k_range_tried": K_RANGE, "k_chosen": K_CHOSEN, "silhouette_chosen": SILHOUETTE_CHOSEN,
                           "min_cluster_fraction": MIN_CLUSTER_FRACTION, "min_cluster_size": MIN_CLUSTER_SIZE,
                           "silhouette_sample_size": SIL_SAMPLE_SIZE,
                           "k_results": [{"k": r["k"], "silhouette": r["silhouette"]} for r in k_results]},
    "feature_names": FEATURE_NAMES,
    "segment_aggregation": seg_agg.to_dict(orient="records"),
    "chi_square_test": {"chi2_statistic": float(chi2_stat), "degrees_of_freedom": int(chi2_dof),
                         "p_value": float(chi2_p), "cramers_v": cramers_v,
                         "cramers_v_ci_95": [V_CI_LOW, V_CI_HIGH], "significant_at_0.05": bool(chi2_p < 0.05)},
    "cross_checks": {
        "vs_risk_tier_cramers_v": CROSS_CRAMERS_V_TIER,
        "vs_bureau_segment_cramers_v": CROSS_CRAMERS_V_BUREAU,
        "vs_repayment_segment_cramers_v": CROSS_CRAMERS_V_REPAYMENT,
        "note": "Descriptive real cross-checks of independence from Problem 1's PD-based tiering and "
                "(when available) Problem 2's bureau behavioral segmentation and Problem 3's repayment "
                "behavior segmentation, not gated pass/fail checks.",
    },
    "statistical_validation": {
        "bootstrap_resamples": N_BOOTSTRAP,
        "validation_checks": {name: bool(ok) for name, ok in validation_checks},
        "failed_validation_checks": _failed_validation_checks,
        "deployment_verdict": ANALYSIS_VERDICT,
        "note": "No monotonicity check in this notebook by design -- behavioral clusters are unordered "
                "categorical segments (see module docstring).",
    },
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": [word_path.name, excel_path.name, html_path.name] + [f"{s}.csv" for s in csv_paths],
    "segment_model_artifact": SEGMENT_MODEL_PATH.name,
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_04_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"[DONE] Mega Project 3 / Notebook 04 complete in {summary['runtime_seconds']}s. "
      f"{K_CHOSEN} real data-driven revolving-credit utilization segments found. "
      f"Statistical robustness verdict: {ANALYSIS_VERDICT}.")
